# Middleware
Middlewre provides a way to more tightly control what happens inside the agent, Middleware is useful for the following:
* Traacking agent behaviour with logging, analytics and debugging,
* Transforming prompts, tool selection and output formatting.
* Adding retries, fall backs and early termination logic.
* Applying rate limits, guardrails and PII detection

In [1]:
#load the API

import os 
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
print("API Key Loaded:", os.getenv("GROQ_API_KEY")[:10] if os.getenv("GROQ_API_KEY") else "FAILED TO LOAD")

#Test the API KEY
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI #alternative 

llm_model = ChatGroq(model ="llama-3.3-70b-versatile" )
print(llm_model.invoke("hello, tell me about yourself in few words").content)


API Key Loaded: gsk_8yEZ0T
I'm an AI assistant, here to help and inform.


# Summarization Middleware
Automatically sumarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
* Long running conversation that execeed context windows.
* Multi turn dialogues with extensive history
* Applications where preserving full conversation context matters.



In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(

    model = llm_model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware( 
            model = llm_model,
            trigger = ("messages",10), #when messages length
            keep=("messages",4) )
    ]
)

In [ ]:
# run with teh thread id
config = {"configurable": {"thread_id": "1" }} #note: always maintian a different thread id for state memory, conversion history

In [14]:
from langchain_core.outputs import chat_generation
from langchain_core.outputs import chat_generation


questions = [
"What is 2+2",
"What is 10*5",
"What is 100/4",
"What is 3*3"

]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content = q)]},config)
    print(f"Messages:{response}")
    print(f"MEssages:{len(response['messages'])}")


Messages:{'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='546b1510-3967-49c9-a939-d182d89ac492'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 41, 'total_tokens': 49, 'completion_time': 0.014047901, 'completion_tokens_details': None, 'prompt_time': 0.002137934, 'prompt_tokens_details': None, 'queue_time': 0.050075216, 'total_time': 0.016185835}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ffb6d-b5e4-7302-ad3d-0e16fb89eb23-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 8, 'total_tokens': 49})]}
MEssages:2
Messages:{'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='546b1510-3967-49c9-a939-d182d89ac492'), AIMessage(co

In [19]:
# GET THE TOKEN SIZE

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """ Search hotels = returns long response to use more tokems"""
    return f""""hotes in {city}:
    1. Grand Hotel - 5 start, $350/night, spa, pool, gym
    2. City Inn - 4 Star, $180/night, business center
    3. Budget Stay - 3start, $75/night, free wifi"""

agent = create_agent(

    model = llm_model,
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model  = llm_model,
            trigger=("tokens",0.05), #or we can set =0.005 i.e.0.5% = ~640 tokens
            keep = ("tokens",0.02)   # 0.2% = ~256 tokens

        )


    ]
)

config = {"configurable":{"thread_id":"test_1"}}

#token counter(approxmiate) function
def count_tokens(messages):
    total_chars = sum(len(str(m.content))for m in messages)
    return total_chars // 4 #4 chars ~ 1 token

In [20]:
#Run test

cities = ["Paries","London","Tokoyo","New York","Singapore"]

for city in cities:
    response = agent.invoke({"messages":[HumanMessage(content = f"Find hotels in {city}")]},
        config = config)

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

    #faction = tokens/12800 #got-40-imini 
    #print(f"{city}: ~{tokens} tokens({fraction: .4%}), {len(response['messages'])} messages")
    #print(response['messages'])

KeyboardInterrupt: 

### Human in the loop
:Pauses agent execution for human approval, editing or rejection of tool calls before they execute. Human-in-the loop is suseful for the following:
High-stakes operations requiring human approval (eg database writes, financial transacitons)
* Compliance workflows where human oversight is mandatory
* Long running coversations where human feedback guides the agent 

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) -> str:
    """ Mock function to read an email by its ID"""
    return f"EMail content for ID {email_id}"

def send_email_tool(recipient: str, subject: str, body:str)->str:
    """ Mock function to send an email"""
    return f"Email send to {recipient} with subject '{subject}'"


In [22]:
agent = create_agent(

    model = llm_model,
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":
                {  "allowed_decisions":["approved","edit","reject"]},
                "read_email_tool":False,
            }

        )

    ]
    
)

In [23]:
config = {"configurable": {"thread_id": "test_approve"}}
#step 1 request
result = agent.invoke(
    {"messages": [HumanMessage(content="send email to john@test.com with subject 'hello' and body 'How are you' ")]},
    config = config
)

result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'hello' and body 'How are you' ", additional_kwargs={}, response_metadata={}, id='274b22fa-8429-46ba-a252-2d7a1bc7ba3a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '98v8ck88s', 'function': {'arguments': '{"body":"How are you","recipient":"john@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 310, 'total_tokens': 342, 'completion_time': 0.067350641, 'completion_tokens_details': None, 'prompt_time': 0.035777946, 'prompt_tokens_details': None, 'queue_time': 0.251600093, 'total_time': 0.103128587}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ffb7b-0ea8-72e1-8a48-9ec4a96b4296-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bo

In [24]:
#Step 2: Approve 
from langgraph.types import Command

if "__interrupt__" in result:
    print(" || Paused! Approveing.....")

    result = agent.invoke(
        Command(resume = {
            "descions": [
                {"type":"approved"}
            ]
        } ),
        config = config
        
    )

print(f"Result: {result['messages'][-1].content}")

 || Paused! Approveing.....


KeyError: 'decisions'

In [26]:
#### Reject
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) -> str:
    """ Mock function to read an email by its ID"""
    return f"EMail content for ID {email_id}"

def send_email_tool(recipient: str, subject: str, body:str)->str:
    """ Mock function to send an email"""
    return f"Email send to {recipient} with subject '{subject}'"


agent = create_agent(

    model = llm_model,
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":
                {  "allowed_decisions":["approved","edit","reject"]},
                "read_email_tool":False,
            }

        )

    ]
    
)

In [27]:
config = {"configurable": {"thread_id":"test-reject"}}

#step 1: Request
result = agent.invoke(

    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you'")]
    },config = config
)

In [ ]:
#step 2: Approve
from langgraph.types import Command

if "__interrupt__" in result:
    print(" || Paused! Approveing.....")

    result = agent.invoke(Command(resume = {
            "descions": [
                {"type":"reject"}]} ), config = config
        
    )

print(f"Result: {result['messages'][-1].content}")

 || Paused! Approveing.....


KeyError: 'decisions'

In [ ]:
# Editing 

# same way we can do for editting